# ptsrv.cli

> Command-line interface for local scan inspection and rendering.

In [ ]:
#| default_exp cli

## CLI: python -m scanplan.cli <command> ...

  * info    scan.e57
  * plan    scan.e57 out.png [--cut 1.2] [--px 0.005] [--style hybrid] [--look up] [--scale 50]
  * section scan.e57 out.png --a x,y --b x,y [--side right] [--depth 1.0] [--px 0.005]
  * elev    scan.e57 out.png --a x,y --b x,y [--side right]

Writes <out>.png plus <out>.pgw (world file) and <out>.json (metadata).


In [ ]:
#| exporti
from __future__ import annotations

In [ ]:
import argparse
import json
import os
import sys
import time

In [ ]:
from ptsrv.cloud import load_cloud
from ptsrv.render import render_plan, render_section, render_elevation

In [ ]:
def _pair(s):
    x, y = (float(v) for v in s.split(","))
    return x, y

In [ ]:
def _write(res, out, scale):
    base = os.path.splitext(out)[0]
    with open(base + ".png", "wb") as f:
        f.write(res.png_bytes(scale))
    with open(base + ".pgw", "w") as f:
        f.write(res.world_file())
    with open(base + ".json", "w") as f:
        json.dump(res.metadata(), f, indent=2)
    w, h = res.image.size
    print(f"wrote {base}.png ({w}x{h}px, {res.pixel_size*1000:g} mm/px), .pgw, .json")

In [ ]:
def main(argv=None):
    p = argparse.ArgumentParser(prog="scanplan")
    p.add_argument("command", choices=["info", "plan", "section", "elev"])
    p.add_argument("e57")
    p.add_argument("out", nargs="?")
    p.add_argument("--align", default="auto", help="auto | 0 | degrees")
    p.add_argument("--max-points", type=int, default=None)
    p.add_argument("--px", type=float, default=0.005)
    p.add_argument("--style", default="hybrid", choices=["color", "depth", "density", "hybrid"])
    p.add_argument("--fill", type=float, default=0.03)
    p.add_argument("--cut-band", type=float, default=0.03)
    p.add_argument("--cut-thicken", type=float, default=0.0)
    p.add_argument("--scale", type=float, default=None, help="print scale denominator for DPI, e.g. 50")
    # plan
    p.add_argument("--cut", type=float, default=1.2)
    p.add_argument("--level", type=float, default=None)
    p.add_argument("--below", type=float, default=None)
    p.add_argument("--look", default="down", choices=["down", "up"])
    p.add_argument("--bounds", default=None, help="xmin,ymin,xmax,ymax")
    # section
    p.add_argument("--a", default=None)
    p.add_argument("--b", default=None)
    p.add_argument("--side", default="right", choices=["left", "right"])
    p.add_argument("--depth", type=float, default=None)
    p.add_argument("--zmin", type=float, default=None)
    p.add_argument("--zmax", type=float, default=None)
    args = p.parse_args(argv)

    align = args.align if args.align == "auto" else float(args.align)
    t = time.time()
    cloud = load_cloud(args.e57, align=align, max_points=args.max_points)
    print(f"loaded {cloud.info.n_points:,} points in {time.time()-t:.2f}s; "
          f"floor z={cloud.info.floor_z:.3f} ceiling z={cloud.info.ceiling_z:.3f} "
          f"rotation={cloud.info.rotation_deg:g} deg", file=sys.stderr)

    if args.command == "info":
        print(json.dumps(cloud.info.to_dict(), indent=2))
        return
    if not args.out:
        p.error("out path required")

    common = dict(pixel_size=args.px, style=args.style, fill_radius=args.fill,
                  cut_thicken=args.cut_thicken)
    t = time.time()
    if args.command == "plan":
        bounds = tuple(float(v) for v in args.bounds.split(",")) if args.bounds else None
        res = render_plan(cloud, cut_height=args.cut, level=args.level, below=args.below,
                          look=args.look, bounds=bounds, cut_band=args.cut_band, **common)
    else:
        if not (args.a and args.b):
            p.error("--a and --b required")
        kw = dict(a=_pair(args.a), b=_pair(args.b), side=args.side, zmin=args.zmin, zmax=args.zmax, **common)
        if args.command == "section":
            res = render_section(cloud, depth=args.depth if args.depth is not None else 1.0,
                                 cut_band=args.cut_band, **kw)
        else:
            res = render_elevation(cloud, depth=args.depth if args.depth is not None else 50.0, **kw)
    print(f"rendered in {time.time()-t:.2f}s", file=sys.stderr)
    _write(res, args.out, args.scale)